# Predictive Maintenance Framework — Nepal... *(placeholder removed)*
# Predictive Maintenance Framework using ML and Real-Time Industrial Analytics

**UWS MSc Data Analytics — Group Project**

This notebook trains the machine-learning core of the Predictive Maintenance Framework on the NASA C-MAPSS turbofan engine degradation dataset (FD001), and exports everything the frontend dashboard needs:

- `rul_model.onnx` — Remaining Useful Life (RUL) regressor, runs client-side in the browser
- `fault_model.onnx` — 3-class health-state classifier (Healthy / Warning / Critical), runs client-side in the browser
- `scaler.json`, `feature_names.json` — preprocessing parameters so the frontend can normalise raw sensor readings identically to training
- `shap_importance.json` — global feature importance for the explainability panel
- `demo_stream.json` — a precomputed, per-cycle time series for a handful of test engines, used to simulate a live sensor feed on the dashboard
- `model_metrics.json` — RMSE / MAE / accuracy for the report and the dashboard's "model performance" panel
- native `.pkl` backups of every model (regressor, classifier, isolation forest, scaler)
- `figures/` — every chart generated during training, saved as high-resolution PNGs for direct use in the dissertation (sensor degradation trend, RUL predicted-vs-actual scatter, confusion matrix, SHAP beeswarm plot, and SHAP feature-importance bar chart)

**How to run:** Runtime → Run all. Takes 2-5 minutes on Colab's free CPU runtime. At the end, a zip file `predictive_maintenance_artifacts.zip` will download automatically — unzip it and copy the contents into the frontend project as instructed in the last cell.

## 1. Setup

In [ ]:
!pip install -q kagglehub xgboost shap onnxmltools onnxruntime skl2onnx onnx

In [ ]:
import os
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import shap

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import IsolationForest
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix

sns.set_style('whitegrid')
np.random.seed(42)

ARTIFACT_DIR = '/content/model_artifacts'
os.makedirs(f'{ARTIFACT_DIR}/models', exist_ok=True)
os.makedirs(f'{ARTIFACT_DIR}/data', exist_ok=True)
os.makedirs(f'{ARTIFACT_DIR}/figures', exist_ok=True)
print('Artifact directory ready:', ARTIFACT_DIR)

## 2. Download the NASA C-MAPSS dataset

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("behrad3d/nasa-cmaps")
print("Path to dataset files:", path)

In [ ]:
# Locate the FD001 files inside the downloaded dataset folder (path can be nested)
def find_file(root, filename):
    for dirpath, _, filenames in os.walk(root):
        if filename in filenames:
            return os.path.join(dirpath, filename)
    raise FileNotFoundError(f'{filename} not found under {root}')

train_path = find_file(path, 'train_FD001.txt')
test_path = find_file(path, 'test_FD001.txt')
rul_path = find_file(path, 'RUL_FD001.txt')

print(train_path)
print(test_path)
print(rul_path)

## 3. Load and structure the data

Each row is one operating cycle for one engine: unit number, cycle number, 3 operational settings, and 21 sensor readings.

In [ ]:
COLS = ['unit', 'cycle', 'op1', 'op2', 'op3'] + [f'sensor{i}' for i in range(1, 22)]

train_df = pd.read_csv(train_path, sep=r'\s+', header=None, names=COLS)
test_df = pd.read_csv(test_path, sep=r'\s+', header=None, names=COLS)
rul_true = pd.read_csv(rul_path, sep=r'\s+', header=None, names=['RUL_true'])

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)
print('Engines in train:', train_df['unit'].nunique())
print('Engines in test:', test_df['unit'].nunique())
train_df.head()

In [ ]:
# Quick look at degradation for a few engines — sensor 11 is a classic degrading sensor in FD001
fig, ax = plt.subplots(figsize=(9, 4))
for unit in train_df['unit'].unique()[:6]:
    sub = train_df[train_df['unit'] == unit]
    ax.plot(sub['cycle'], sub['sensor11'], alpha=0.8, label=f'Engine {unit}')
ax.set_xlabel('Cycle')
ax.set_ylabel('Sensor 11 reading')
ax.set_title('Sensor 11 trend across engine lifetimes (degradation signal)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f'{ARTIFACT_DIR}/figures/bonus_sensor_degradation_trend.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Label engineering — Remaining Useful Life (RUL)

For each engine in the training set we know the full run-to-failure trajectory, so RUL at any cycle = (final cycle of that engine) − (current cycle). RUL is clipped at 125 cycles, a standard practice for this dataset: an engine 200 cycles from failure is operationally no different from one 125 cycles from failure, and clipping stops the model from being penalised for large errors on healthy engines it cannot meaningfully predict.

In [ ]:
RUL_CLIP = 125

max_cycle = train_df.groupby('unit')['cycle'].transform('max')
train_df['RUL'] = (max_cycle - train_df['cycle']).clip(upper=RUL_CLIP)

train_df[['unit', 'cycle', 'RUL']].head(10)

## 5. Feature engineering

Raw sensor readings are noisy. We add rolling mean and rolling standard deviation (window = 5 cycles) per engine, per sensor — this captures the *trend* and *volatility* of degradation, not just the instantaneous value, which is what actually drives RUL and fault predictions.

In [ ]:
WINDOW = 5
sensor_cols = [c for c in train_df.columns if c.startswith('sensor')]

def add_rolling_features(df, sensor_cols, window=WINDOW):
    df = df.sort_values(['unit', 'cycle']).copy()
    for c in sensor_cols:
        df[f'{c}_roll_mean'] = df.groupby('unit')[c].transform(lambda s: s.rolling(window, min_periods=1).mean())
        df[f'{c}_roll_std'] = df.groupby('unit')[c].transform(lambda s: s.rolling(window, min_periods=1).std().fillna(0))
    return df

train_feat = add_rolling_features(train_df, sensor_cols)
test_feat = add_rolling_features(test_df, sensor_cols)

# Drop sensors that are constant (zero variance) in the training set — they carry no information
variances = train_feat[sensor_cols].var()
low_variance_sensors = variances[variances < 1e-6].index.tolist()
print('Dropping low-variance sensors:', low_variance_sensors)

useful_sensor_cols = [c for c in sensor_cols if c not in low_variance_sensors]
feature_cols = useful_sensor_cols + ['op1', 'op2', 'op3'] + \
    [f'{c}_roll_mean' for c in useful_sensor_cols] + [f'{c}_roll_std' for c in useful_sensor_cols]

print(f'Final feature count: {len(feature_cols)}')

### Figure 1 — feature engineering pipeline diagram

This is a conceptual diagram (not a data plot) summarising the transformation this section just performed, for direct use as Figure 1 in the dissertation.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.2))
ax.set_xlim(0, 12)
ax.set_ylim(0, 3)
ax.axis('off')

stages = [
    ('Raw sensor readings\n(21 channels + 3 op. settings)', 0.2),
    ('Drop low-variance\nsensors (6 removed)', 2.75),
    ('Rolling mean & std\n(5-cycle window)', 5.3),
    ('Min-max scaling\n(fit on train set)', 7.85),
    (f'Final feature vector\n({len(feature_cols)} dimensions)', 10.4),
]

box_w, box_h = 2.15, 1.6
for text, x in stages:
    box = plt.Rectangle((x, 0.7), box_w, box_h, facecolor='#FFE9BE', edgecolor='#B47C1B', linewidth=1.5, zorder=2)
    ax.add_patch(box)
    ax.text(x + box_w / 2, 0.7 + box_h / 2, text, ha='center', va='center', fontsize=9.5, zorder=3)

for _, x in stages[:-1]:
    ax.annotate('', xy=(x + box_w + 0.4, 1.5), xytext=(x + box_w, 1.5),
                arrowprops=dict(arrowstyle='-|>', color='#7A4E00', linewidth=1.8))

ax.set_title('Feature engineering pipeline: raw sensor readings to final scaled feature vector', fontsize=12, pad=12)
plt.tight_layout()
plt.savefig(f'{ARTIFACT_DIR}/figures/figure_01_feature_engineering_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Scale features and save preprocessing parameters for the frontend

In [ ]:
scaler = MinMaxScaler()
X_train_full = scaler.fit_transform(train_feat[feature_cols]).astype(np.float32)
y_rul_full = train_feat['RUL'].values.astype(np.float32)

# Save scaler parameters as JSON so the browser can apply the identical (x - min) / (max - min) transform
scaler_json = {
    'feature_order': feature_cols,
    'data_min': scaler.data_min_.tolist(),
    'data_max': scaler.data_max_.tolist(),
}
with open(f'{ARTIFACT_DIR}/data/scaler.json', 'w') as f:
    json.dump(scaler_json, f)

with open(f'{ARTIFACT_DIR}/data/feature_names.json', 'w') as f:
    json.dump({'features': feature_cols, 'window': WINDOW, 'rul_clip': RUL_CLIP}, f)

print('Saved scaler.json and feature_names.json')

## 7. Health-state labels for classification

We bin RUL into three operational states used throughout the dashboard:

| State | RUL range | Meaning |
|---|---|---|
| 0 — Healthy | RUL > 90 | normal operation |
| 1 — Warning | 30 < RUL ≤ 90 | schedule inspection |
| 2 — Critical | RUL ≤ 30 | prioritise maintenance |

In [ ]:
def health_state(rul):
    if rul > 90:
        return 0
    elif rul > 30:
        return 1
    return 2

HEALTH_LABELS = {0: 'Healthy', 1: 'Warning', 2: 'Critical'}

train_feat['health_state'] = train_feat['RUL'].apply(health_state)
y_cls_full = train_feat['health_state'].values.astype(np.int64)

train_feat['health_state'].value_counts().rename(index=HEALTH_LABELS)

## 8. Train / validation split — grouped by engine

A random row-level split would leak cycles from the same engine into both train and validation, inflating performance. `GroupShuffleSplit` keeps every engine's full trajectory on one side only.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(X_train_full, y_rul_full, groups=train_feat['unit']))

X_train, X_val = X_train_full[train_idx], X_train_full[val_idx]
y_train, y_val = y_rul_full[train_idx], y_rul_full[val_idx]
yc_train, yc_val = y_cls_full[train_idx], y_cls_full[val_idx]

print(f'Train rows: {X_train.shape[0]} | Validation rows: {X_val.shape[0]}')

## 9. Train the RUL regressor (XGBoost)

In [ ]:
rul_model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)
rul_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

pred_val = rul_model.predict(X_val)
rmse = mean_squared_error(y_val, pred_val) ** 0.5
mae = mean_absolute_error(y_val, pred_val)
r2 = r2_score(y_val, pred_val)
print(f'RUL model — RMSE: {rmse:.2f} | MAE: {mae:.2f} | R2: {r2:.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_val, pred_val, alpha=0.3, s=12)
ax.plot([0, RUL_CLIP], [0, RUL_CLIP], 'r--', linewidth=1)
ax.set_xlabel('Actual RUL')
ax.set_ylabel('Predicted RUL')
ax.set_title('RUL regressor — predicted vs actual (validation set)')
plt.tight_layout()
plt.savefig(f'{ARTIFACT_DIR}/figures/figure_02_07_rul_predicted_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Train the health-state classifier (XGBoost)

In [ ]:
fault_model = xgb.XGBClassifier(
    n_estimators=250,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    num_class=3,
    random_state=42,
)
fault_model.fit(X_train, yc_train, eval_set=[(X_val, yc_val)], verbose=False)

pred_cls = fault_model.predict(X_val)
acc = accuracy_score(yc_val, pred_cls)
print(f'Health-state classifier accuracy: {acc:.3f}')
print(classification_report(yc_val, pred_cls, target_names=list(HEALTH_LABELS.values())))

In [ ]:
cm = confusion_matrix(yc_val, pred_cls)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=list(HEALTH_LABELS.values()), yticklabels=list(HEALTH_LABELS.values()), ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Health-state confusion matrix')
plt.tight_layout()
plt.savefig(f'{ARTIFACT_DIR}/figures/figure_03_08_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Anomaly detection (Isolation Forest)

An unsupervised model flags sensor patterns that don't fit the learned "normal operation" envelope — this catches unusual behaviour the two supervised models weren't explicitly trained to recognise, and feeds the alerts panel on the dashboard.

In [ ]:
iso_forest = IsolationForest(n_estimators=200, contamination=0.05, random_state=42)
iso_forest.fit(X_train_full)

raw_scores = iso_forest.decision_function(X_train_full)
# Normalise so higher = more anomalous, scaled 0-1 for the frontend
anomaly_min, anomaly_max = raw_scores.min(), raw_scores.max()
def normalise_anomaly(raw):
    return float(np.clip(1 - (raw - anomaly_min) / (anomaly_max - anomaly_min + 1e-9), 0, 1))

print('Isolation forest trained. Example anomaly scores:', [normalise_anomaly(s) for s in raw_scores[:5]])

## 12. Explainability — global SHAP feature importance

We compute SHAP values for the RUL model on a validation sample and save the top contributing features. The dashboard's explainability panel uses this to show *which sensors* are driving the fleet's predictions, in plain terms rather than as a black box.

In [ ]:
explainer = shap.TreeExplainer(rul_model)
sample_idx = np.random.choice(X_val.shape[0], size=min(300, X_val.shape[0]), replace=False)
shap_values = explainer.shap_values(X_val[sample_idx])

mean_abs_shap = np.abs(shap_values).mean(axis=0)
importance_order = np.argsort(mean_abs_shap)[::-1]

TOP_N = 12
shap_importance = [
    {'feature': feature_cols[i], 'importance': float(mean_abs_shap[i])}
    for i in importance_order[:TOP_N]
]

with open(f'{ARTIFACT_DIR}/data/shap_importance.json', 'w') as f:
    json.dump(shap_importance, f, indent=2)

for row in shap_importance:
    print(f"{row['feature']:<24} {row['importance']:.3f}")

In [ ]:
shap.summary_plot(shap_values, X_val[sample_idx], feature_names=feature_cols, show=False, max_display=12)
plt.tight_layout()
plt.savefig(f'{ARTIFACT_DIR}/figures/bonus_shap_summary_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

The beeswarm plot above shows the full distribution of each feature's effect; for the dashboard's explainability panel (and for the dissertation's global feature-importance figure) we use a simpler horizontal bar chart of the same `shap_importance` ranking, styled to match what actually renders in the browser.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
bar_features = [row['feature'] for row in shap_importance][::-1]
bar_values = [row['importance'] for row in shap_importance][::-1]
ax.barh(bar_features, bar_values, color='#FFB020')
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('Global SHAP feature importance — RUL regressor')
plt.tight_layout()
plt.savefig(f'{ARTIFACT_DIR}/figures/figure_04_09_shap_feature_importance_bar.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Export models to ONNX

ONNX lets the trained XGBoost models run **directly in the browser** via `onnxruntime-web` — no Python backend needed for inference. This is what makes the frontend a genuine client-side ML dashboard rather than a form that calls an API.

In [ ]:
from onnxmltools import convert_xgboost
from onnxmltools.convert.common.data_types import FloatTensorType
import onnxruntime as ort

n_features = X_train_full.shape[1]
initial_type = [('input', FloatTensorType([None, n_features]))]

onnx_rul = convert_xgboost(rul_model, initial_types=initial_type)
with open(f'{ARTIFACT_DIR}/models/rul_model.onnx', 'wb') as f:
    f.write(onnx_rul.SerializeToString())

onnx_fault = convert_xgboost(fault_model, initial_types=initial_type)
with open(f'{ARTIFACT_DIR}/models/fault_model.onnx', 'wb') as f:
    f.write(onnx_fault.SerializeToString())

print('ONNX export complete.')

In [ ]:
# Sanity check: ONNX Runtime predictions must match the native XGBoost predictions
sess_rul = ort.InferenceSession(f'{ARTIFACT_DIR}/models/rul_model.onnx')
onnx_pred = sess_rul.run(None, {'input': X_val[:5]})[0].flatten()
native_pred = rul_model.predict(X_val[:5])
print('ONNX RUL prediction:  ', onnx_pred)
print('Native RUL prediction:', native_pred)
print('Max abs difference:', np.max(np.abs(onnx_pred - native_pred)))

sess_fault = ort.InferenceSession(f'{ARTIFACT_DIR}/models/fault_model.onnx')
onnx_out = sess_fault.run(None, {'input': X_val[:5]})
print('\nONNX classifier input name:', sess_fault.get_inputs()[0].name)
print('ONNX classifier output names:', [o.name for o in sess_fault.get_outputs()])
print('ONNX classifier labels:', onnx_out[0])
print('Native classifier labels:', fault_model.predict(X_val[:5]))

## 14. Build the live-stream demo dataset

The frontend doesn't have a live sensor feed — it's a student prototype. Instead we precompute the full cycle-by-cycle prediction history for a handful of *test-set* engines (data the models never trained on) and export it as JSON. The dashboard replays this at a fixed interval to *simulate* a live telemetry stream, exactly like the group's shared architecture describes.

In [ ]:
N_DEMO_ENGINES = 10
demo_units = sorted(test_feat['unit'].unique())[:N_DEMO_ENGINES]

demo_rows = []
key_sensors_for_chart = [c for c in ['sensor2', 'sensor3', 'sensor4', 'sensor7', 'sensor11', 'sensor15'] if c in useful_sensor_cols]

for unit in demo_units:
    sub = test_feat[test_feat['unit'] == unit].sort_values('cycle')
    X_unit = scaler.transform(sub[feature_cols]).astype(np.float32)

    rul_pred = sess_rul.run(None, {'input': X_unit})[0].flatten()
    fault_out = sess_fault.run(None, {'input': X_unit})
    health_pred = fault_out[0]
    health_proba = fault_out[1]
    anomaly_raw = iso_forest.decision_function(X_unit)

    for i, (_, row) in enumerate(sub.iterrows()):
        demo_rows.append({
            'unit': int(unit),
            'cycle': int(row['cycle']),
            'sensors': {c: round(float(row[c]), 3) for c in key_sensors_for_chart},
            # Raw (unscaled) feature vector in feature_cols order — the frontend scales this itself
            # using scaler.json and runs it through the ONNX models directly in the browser. This is
            # what makes the dashboard genuine client-side inference rather than a replay of numbers
            # computed here in Python; the precomputed fields below act as a reference / fallback.
            'features': [round(float(v), 5) for v in row[feature_cols].values],
            'predicted_RUL': round(float(rul_pred[i]), 1),
            'health_state': int(health_pred[i]),
            'health_label': HEALTH_LABELS[int(health_pred[i])],
            'health_confidence': round(float(max(health_proba[i])), 3),
            'anomaly_score': round(normalise_anomaly(anomaly_raw[i]), 3),
            'is_anomaly': bool(normalise_anomaly(anomaly_raw[i]) > 0.6),
        })

with open(f'{ARTIFACT_DIR}/data/demo_stream.json', 'w') as f:
    json.dump(demo_rows, f)

print(f'Demo stream built: {len(demo_rows)} rows across {len(demo_units)} engines')
print('Key sensors used for charting:', key_sensors_for_chart)

## 15. Save model performance summary and native `.pkl` backups

In [ ]:
import pickle

model_metrics = {
    'rul_model': {'rmse': round(float(rmse), 2), 'mae': round(float(mae), 2), 'r2': round(float(r2), 3)},
    'fault_model': {'accuracy': round(float(acc), 3)},
    'dataset': 'NASA C-MAPSS FD001',
    'n_train_engines': int(train_df['unit'].nunique()),
    'n_test_engines': int(test_df['unit'].nunique()),
    'n_features': int(len(feature_cols)),
    'rul_clip': RUL_CLIP,
}
with open(f'{ARTIFACT_DIR}/data/model_metrics.json', 'w') as f:
    json.dump(model_metrics, f, indent=2)

with open(f'{ARTIFACT_DIR}/models/rul_model.pkl', 'wb') as f:
    pickle.dump(rul_model, f)
with open(f'{ARTIFACT_DIR}/models/fault_model.pkl', 'wb') as f:
    pickle.dump(fault_model, f)
with open(f'{ARTIFACT_DIR}/models/isolation_forest.pkl', 'wb') as f:
    pickle.dump(iso_forest, f)
with open(f'{ARTIFACT_DIR}/models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('Saved model_metrics.json and .pkl backups.')
print(json.dumps(model_metrics, indent=2))

## 16. Package everything into a zip and download

In [ ]:
zip_path = '/content/predictive_maintenance_artifacts.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(ARTIFACT_DIR):
        for file in files:
            full_path = os.path.join(root, file)
            arcname = os.path.relpath(full_path, ARTIFACT_DIR)
            zf.write(full_path, arcname)

print('Zip created at:', zip_path)
print('Contents:')
with zipfile.ZipFile(zip_path) as zf:
    for name in zf.namelist():
        print(' -', name)

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print('Not running in Colab — find the zip at', zip_path)

## 17. Where these files go

Unzip `predictive_maintenance_artifacts.zip`. It contains a `models/` folder, a `data/` folder, and a `figures/` folder.

**Frontend** — copy `models/` and `data/` into the frontend project like this:

```
frontend/
  public/
    models/
      rul_model.onnx        <-  from models/rul_model.onnx
      fault_model.onnx      <-  from models/fault_model.onnx
    data/
      scaler.json            <-  from data/scaler.json
      feature_names.json     <-  from data/feature_names.json
      shap_importance.json   <-  from data/shap_importance.json
      demo_stream.json       <-  from data/demo_stream.json
      model_metrics.json     <-  from data/model_metrics.json
```

The `.pkl` files (`rul_model.pkl`, `fault_model.pkl`, `isolation_forest.pkl`, `scaler.pkl`) are not used by the frontend — keep them for the dissertation appendix / methodology section as evidence of the native trained models, or for any further Python-side analysis.

**Dissertation** — every filename in `figures/` is named after the exact dissertation figure number it belongs to, so you can find each one instantly:

| File in `figures/` | Dissertation figure |
|---|---|
| `figure_01_feature_engineering_pipeline.png` | **Figure 1** |
| `figure_02_07_rul_predicted_vs_actual.png` | **Figure 2** (Implementation) *and* **Figure 7** (Results) — same image, used in both places |
| `figure_03_08_confusion_matrix.png` | **Figure 3** (Implementation) *and* **Figure 8** (Results) |
| `figure_04_09_shap_feature_importance_bar.png` | **Figure 4** (Implementation) *and* **Figure 9** (Results) |
| `bonus_sensor_degradation_trend.png` | not a numbered figure — optional extra EDA evidence |
| `bonus_shap_summary_beeswarm.png` | not a numbered figure — optional richer explainability chart |

**Figures 5, 6, and 10 cannot come from this notebook** — they are screenshots of the *running frontend dashboard*, which only exists once `npm run dev` is up in your browser. When you take them, name them the same way so they're just as easy to find:

| Screenshot to take | Save as |
|---|---|
| Fleet overview page | `figure_05_fleet_overview.png` |
| Engine detail page | `figure_06_engine_detail.png` |
| Full dashboard (fleet + alerts + explainability visible together) | `figure_10_full_dashboard.png` |

Once the files are in place, `cd frontend && npm install && npm run dev` — see the frontend README for full setup.